# 分布式通信演示

第 6 章完成了 SFT 变长注意力优化。本章暂时离开完整训练，固定使用当前环境中的 **Qwen3-1.7B、bf16 和两张 Ascend NPU**，集中回答一个问题：

> 给定张量规模和并行方式，一次通信要交换多少数据？PyTorch 如何发起通信？TorchTitan 又在哪里实现它？

本章不比较 loss 或端到端吞吐，只通过最小演示建立 FSDP、TP 和 CP 的通信账本。

---


## 教程进度回顾

| 章节 | 内容 | 状态 |
|---|---|---|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| 第 3 章 | 数据准备 + 基线训练 + 推理评测 | ✅ 已完成 |
| 第 4 章 | 融合算子优化 + Profiling | ✅ 已完成 |
| 第 5 章 | Attention 公式、算子与 TorchTitan dispatch | ✅ 已完成 |
| 第 6 章 | Sequence Packing、VarLen Attention 与端到端效率 | ✅ 已完成 |
| **第 7 章** | **FSDP、TP、CP 集合通信与通信量分析** | ← 当前 |

---


## 本章目标

完成本章后，你将能够：

- 根据 Qwen3 的张量形状和 dtype 计算通信数据量；
- 区分每个 rank 的本地数据量、跨卡交换量和聚合数据量；
- 用 `torch.distributed` 构造 FSDP、TP、CP 中的代表性集合通信；
- 通过逐 rank 重复计时观察延迟分布，并按需采集 Ascend Profiler trace 核对 HCCL 调用；
- 把演示中的调用对应到 TorchTitan 的 mesh、并行化计划和 FSDP/CP 实现。

本章只做最小运行检查。EP 不在本章范围内：Qwen3-1.7B 是稠密模型，当前两卡配置也没有启用专家并行。

---


## 前置条件

- 完成第 2 章，当前环境能够正常运行 `torch.distributed` 和 `torch_npu`；
- 完成第 4 章，理解 wall-time、重复计时与 profiler trace 的分工；
- 完成第 6 章，熟悉 Qwen3 attention 的主要张量形状与 VarLen 数据布局；
- 运行集合通信单元时，当前环境可使用两张 Ascend NPU 启动双进程任务。纯公式、账本和练习单元可在 CPU/离线环境完成。

---


## 本章结构

| Notebook | 具体问题 | 主要集合通信 |
|---|---|---|
| [07.01](07.01_chapter_intro.ipynb) | 章节介绍（本节） | — |
| [07.02](07.02_fsdp_collectives.ipynb) | FSDP 如何恢复完整参数，再切回梯度分片？ | `all_gather`、`reduce_scatter` |
| [07.03](07.03_tp_collectives.ipynb) | TP 的列切分和行切分分别在哪里需要同步？ | `all_reduce` |
| [07.04](07.04_cp_collectives.ipynb) | CP 如何在 attention 前后交换 sequence/head 布局？ | `all_to_all_single` |
| [07.05](07.05_communication_volume_and_overlap.ipynb) | 如何汇总 FSDP/TP/CP 整步通信量，并判断 overlap 的真实收益？ | 跨模式通信账本 |
| [07.06](07.06_chapter_practice.ipynb) | 使用同一组 Qwen3 参数完成章末练习 | — |

前三个演示按照同一顺序展开：**形状 → 数据归属 → 字节数 → PyTorch 调用 → 性能分析 → TorchTitan 实现**。07.05 汇总训练步成本，07.06 完成章节练习。

---


## 固定实验卡

| 项目 | 数值 |
|---|---:|
| 隐藏维度 | 2,048 |
| 中间维度 | 6,144 |
| Transformer 层数 | 28 |
| attention heads / KV heads | 16 / 8 |
| head dim | 128 |
| batch × 序列长度 | 2 × 4,096 |
| 数据类型 | bf16（2 bytes） |
| 通信并行度 | 2 |

这里的 MB 使用十进制（`1 MB = 1,000,000 bytes`），便于和 Ascend Profiler CSV 中的 `Transit Size(MB)` 对照。


In [ ]:
# 这段算术是三个 Demo 共用的事实来源；后续 notebook 会重复需要的部分。
B, S, H, I, D = 2, 4096, 2048, 6144, 128
N, BYTES = 2, 2
H_Q, H_KV = 16, 8

def mb(n: int) -> float:
    return n / 1_000_000

# 2*H 是 block 的两个 RMSNorm；2*D 是 Q/K 的 head-dim RMSNorm。
layer_params = H * (H_Q * D) + 2 * H * (H_KV * D) + H * (H_Q * D) + 3 * H * I + 2 * H + 2 * D
layer_full = layer_params * BYTES
activation = B * S * H * BYTES

{
    'transformer_layer_parameters': layer_params,
    'layer_full_MB': round(mb(layer_full), 3),
    'layer_shard_MB_at_degree_2': round(mb(layer_full / N), 3),
    'hidden_activation_MB': round(mb(activation), 3),
}


---

## 统一的通信账本

对每个演示，我们同时记录三类数据：

1. **每个 rank 的逻辑张量字节数**：集合通信的输入或输出张量有多大；
2. **每个 rank 的跨卡交换字节数**：两张卡之间真正需要发送和接收多少 payload；
3. **实测时间**：完成若干次预热后，保存每次、每个 rank 的同步 wall-time；以同一 iteration 的较慢 rank 作为该次 collective 延迟，并报告 median、P95、范围和 rank 差异。

wall-time JSON 是统计来源；传入 `--profile-dir` 时，脚本会另采一轮 trace 检查 HCCL 调用。trace 不参与上面的 wall-time 统计。

不要把这三个数字混成一个笼统的“通信成本”。例如，FSDP all-gather 的输入是 50.3 MB 参数分片，输出是 100.7 MB 完整参数，而每张卡真正需要从对端接收的是 50.3 MB。

---


## 运行约定

真正的集合通信需要两个进程。可直接在本目录运行已经准备好的脚本，不需要从 Notebook 手动复制代码：

```bash
torchrun --standalone --nproc_per_node=2 scripts/fsdp_collectives.py
torchrun --standalone --nproc_per_node=2 scripts/tp_collectives.py
torchrun --standalone --nproc_per_node=2 scripts/cp_collectives.py
```

命令不覆盖 `ASCEND_VISIBLE_DEVICES`。在动态资源环境中，应由调度器预先暴露分配到的两张卡；脚本只使用 `LOCAL_RANK=0,1` 选择可见设备，不能假设物理卡号一定是 0 和 1。

三个脚本默认预热 5 次、测量 20 次。rank 0 汇总所有 rank 的逐 iteration 样本，报告各 rank 与慢 rank 路径的 median/P95/range，并可用 `--output-json` 保存原始样本。传入 `--profile-dir` 才会额外采集 trace。使用 `--help` 可以修改参数；Notebook 中的算术单元可直接在 CPU 运行。


## 练习

1. （判断题）第 7 章的独立 collective demo 主要建立通信账本，不直接比较完整训练的 loss 或吞吐。

2. （单选题）本章为了和 Ascend Profiler 的 Transit Size 对照，MB 采用哪种定义？
    A. 1 MB = 1,000,000 bytes
    B. 1 MB = 1,048,576 bytes
    C. 1 MB = 1,024 bytes
    D. 1 MB = 1,000 bytes

3. （判断题）一次 collective 的延迟应只取 rank 0 的时间，无需观察其他 rank。

4. （多选题）本章统一区分哪些通信口径？
    A. rank 的逻辑输入/输出张量字节数
    B. rank 的跨卡交换 payload
    C. 多次运行的慢 rank latency 分布
    D. 单个最快 kernel 的耗时

In [ ]:
!cat ./answer/07.01_answer.txt
